In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/lthnhcng/imbd-datasets/IMDB Dataset.csv


In [2]:
import random, platform
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import transformers
from transformers import AutoTokenizer

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MODEL_NAME = "bert-base-uncased"  
print("transformers:", transformers.__version__)
print("torch       :", torch.__version__)
print("python      :", platform.python_version())
torch.set_num_threads(max(1, torch.get_num_threads()))
print("device      : CPU")
print("CPU threads :", torch.get_num_threads())

transformers: 5.20.2
torch       : 2.11.0+cpu
python      : 3.13.18
device      : CPU
CPU threads : 2


# Load tokenizer

In [3]:


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(type(tokenizer).__name__)
print("Vocab size:", tokenizer.vocab_size)
print("Special tokens:", tokenizer.all_special_tokens)
print("Special token IDs:", tokenizer.all_special_ids)
print("Max model length:", tokenizer.model_max_length)



config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BertTokenizer
Vocab size: 30522
Special tokens: ['[UNK]', '[SEP]', '[PAD]', '[CLS]', '[MASK]']
Special token IDs: [100, 102, 0, 101, 103]
Max model length: 512


# Calcualte exer2

In [4]:
text = "The cats are playing."

tokens = tokenizer(text, padding=True, truncation=True, return_tensors="pt")

# (a) keys của output
print("tokens.keys():", list(tokens.keys()))
print()
for k, v in tokens.items():
    print(f"{k:16s} shape={tuple(v.shape)}  ->  {v.tolist()}")

tokens.keys(): ['input_ids', 'token_type_ids', 'attention_mask']

input_ids        shape=(1, 7)  ->  [[101, 1996, 8870, 2024, 2652, 1012, 102]]
token_type_ids   shape=(1, 7)  ->  [[0, 0, 0, 0, 0, 0, 0]]
attention_mask   shape=(1, 7)  ->  [[1, 1, 1, 1, 1, 1, 1]]


In [5]:
ids = tokens["input_ids"][0].tolist()
mask = tokens["attention_mask"][0].tolist()
tok_seq = tokenizer.convert_ids_to_tokens(ids)

df = pd.DataFrame({
    "position": range(len(ids)),
    "token": tok_seq,
    "token_id": ids,
    "attention_mask": mask,
    "is_special": [t in tokenizer.all_special_tokens for t in tok_seq],
})
df

,position,token,token_id,attention_mask,is_special
0,0,[CLS],101,1,True
1,1,the,1996,1,False
2,2,cats,8870,1,False
3,3,are,2024,1,False
4,4,playing,2652,1,False
5,5,.,1012,1,False
6,6,[SEP],102,1,True


In [6]:
# input ids and attention mask


print("decode (giữ special) :", tokenizer.decode(ids))
print("decode (bỏ special)  :", tokenizer.decode(ids, skip_special_tokens=True))
print()
print("convert_tokens_to_ids('[CLS]') =", tokenizer.convert_tokens_to_ids("[CLS]"))
print("convert_tokens_to_ids('[SEP]') =", tokenizer.convert_tokens_to_ids("[SEP]"))
print("convert_tokens_to_ids('[PAD]') =", tokenizer.convert_tokens_to_ids("[PAD]"))
print("convert_tokens_to_ids('[MASK]')=", tokenizer.convert_tokens_to_ids("[MASK]"))



decode (giữ special) : [CLS] the cats are playing. [SEP]
decode (bỏ special)  : the cats are playing.

convert_tokens_to_ids('[CLS]') = 101
convert_tokens_to_ids('[SEP]') = 102
convert_tokens_to_ids('[PAD]') = 0
convert_tokens_to_ids('[MASK]')= 103


In [7]:
# subword
words = [
    "cat", "cats", "playing", "unbelievable", "unbelievably",
    "tokenization", "pretraining", "electroencephalography",
    "COVID-19", "gooooood", "asdfghjk", "2026", "Hanoi", "phở",
]
rows = []
for w in words:
    pieces = tokenizer.tokenize(w)
    rows.append({
        "word": w,
        "tokens": pieces,
        "#tokens": len(pieces),
        "has_[UNK]": tokenizer.unk_token in pieces,
    })
pd.DataFrame(rows)

,word,tokens,#tokens,has_[UNK]
0,cat,[cat],1,False
1,cats,[cats],1,False
2,playing,[playing],1,False
3,unbelievable,[unbelievable],1,False
4,unbelievably,"[un, ##bel, ##ie, ##va, ##bly]",5,False
5,tokenization,"[token, ##ization]",2,False
6,pretraining,"[pre, ##train, ##ing]",3,False
7,electroencephalography,"[electro, ##ence, ##pha, ##log, ##raphy]",5,False
8,COVID-19,"[co, ##vid, -, 19]",4,False
9,gooooood,"[goo, ##oo, ##ood]",3,False


In [8]:
# sepcail token
single = tokenizer("this movie is great")
pair   = tokenizer("sentence A", "sentence B")

for name, enc in [("Single", single), ("Pair", pair)]:
    toks = tokenizer.convert_ids_to_tokens(enc["input_ids"])
    print(f"{name}")
    print("tokens        :", toks)
    print("token_type_ids:", enc.get("token_type_ids"))
    print("attention_mask:", enc["attention_mask"])
    print()

Single
tokens        : ['[CLS]', 'this', 'movie', 'is', 'great', '[SEP]']
token_type_ids: [0, 0, 0, 0, 0, 0]
attention_mask: [1, 1, 1, 1, 1, 1]

Pair
tokens        : ['[CLS]', 'sentence', 'a', '[SEP]', 'sentence', 'b', '[SEP]']
token_type_ids: [0, 0, 0, 0, 1, 1, 1]
attention_mask: [1, 1, 1, 1, 1, 1, 1]



In [9]:
# padding and attention mask


batch = [
    "The cat eats fish.",
    "I really did not enjoy this film at all, although the acting was decent.",
    "Great!",
]
enc = tokenizer(batch, padding=True, truncation=True, return_tensors="pt")

print("input_ids shape     :", tuple(enc["input_ids"].shape))
print("attention_mask shape:", tuple(enc["attention_mask"].shape))
print()
for i, s in enumerate(batch):
    toks = tokenizer.convert_ids_to_tokens(enc["input_ids"][i])
    print(f"[{i}] {s}")
    print("    tokens:", toks)
    print("    mask  :", enc["attention_mask"][i].tolist())
    print()



input_ids shape     : (3, 18)
attention_mask shape: (3, 18)

[0] The cat eats fish.
    tokens: ['[CLS]', 'the', 'cat', 'eats', 'fish', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
    mask  : [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

[1] I really did not enjoy this film at all, although the acting was decent.
    tokens: ['[CLS]', 'i', 'really', 'did', 'not', 'enjoy', 'this', 'film', 'at', 'all', ',', 'although', 'the', 'acting', 'was', 'decent', '.', '[SEP]']
    mask  : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

[2] Great!
    tokens: ['[CLS]', 'great', '!', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
    mask  : [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

